# exp02 — Exp01 QLoRA: inference-only diagnostics (no training)

Loads Qwen3-8B-Base (4-bit NF4) exactly as the exp01 training notebook did, with no
optimizer and no `trainer.train()`. Two passes over all labelled rows (train + held-out val):

1. **Step-0 / pre-training model**: base model + the freshly initialised `score` head
   (`set_seed(42)` before `from_pretrained`, same as exp01). LoRA `B` is zero-initialised,
   so this is the function exp01 started from. Measures step-0 cross-entropy and logit scale.
2. **Exp01 trained model**: the saved adapter + `score` head from
   `ritisingh123/engs108-qwen3-8b-lora-adapter`, loaded with `is_trainable=False`, `model.eval()`.

Writes per-row logits for both passes, pooled final hidden states, and `diagnostics.json`.
Internet is disabled; `bitsandbytes` comes from the attached wheel dataset.

In [ ]:
import glob
import subprocess
import sys

wheels = sorted(glob.glob("/kaggle/input/**/bitsandbytes-*.whl", recursive=True))
assert wheels, "bitsandbytes wheel not found; attach the engs108-offline-wheels dataset"
print("Installing", wheels[-1])
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheels[-1]],
    check=True,
)

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import time
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

RUN_START = time.time()

DATA_DIR = "/kaggle/input/competitions/name-that-llm"
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1"
OUTPUT_DIR = "/kaggle/working"
MAX_LENGTH = 512
BATCH_SIZE = 8
SEED = 42
VAL_SIZE = 0.15
N_SIM_HEADS = 10  # extra random score-head draws to check the step-0 loss is not seed-specific

adapter_configs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert len(adapter_configs) == 1, adapter_configs
ADAPTER_DIR = os.path.dirname(adapter_configs[0])
print("Adapter dir:", ADAPTER_DIR, sorted(os.listdir(ADAPTER_DIR)))

versions = {p: metadata.version(p) for p in
            ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "scikit-learn"]}
print(versions)
print("CUDA available:", torch.cuda.is_available(), "| visible GPUs:", torch.cuda.device_count())

## Recreate the exp01 split (identical code path) and verify it

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")

with open(f"{ADAPTER_DIR}/label_map.json") as f:
    id2label = {int(k): v for k, v in json.load(f)["id2label"].items()}
label2id = {v: k for k, v in id2label.items()}
model_names = [id2label[i] for i in range(len(id2label))]
assert model_names == sorted(train["model"].unique())

# Same as exp01: df = train[[...]].copy(); GroupShuffleSplit on prompt_id over df.
df = train[["id", "prompt_id", "text", "model"]].copy()
df["labels"] = df["model"].map(label2id).astype(int)
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(splitter.split(df, groups=df["prompt_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

EXP01_VAL_PROMPTS = ['p009', 'p012', 'p018', 'p019', 'p026', 'p029', 'p031', 'p036', 'p055',
                     'p056', 'p064', 'p068', 'p069', 'p073', 'p076', 'p078', 'p082', 'p104',
                     'p110', 'p118', 'p127', 'p132', 'p141']
assert sorted(val_df["prompt_id"].unique()) == EXP01_VAL_PROMPTS
assert not set(train_df["prompt_id"]) & set(val_df["prompt_id"])
assert (len(train_df), train_df["prompt_id"].nunique()) == (2285, 127)
assert (len(val_df), val_df["prompt_id"].nunique()) == (414, 23)
assert (val_df["model"].value_counts() == 23).all() and val_df["model"].nunique() == 18
print("Split verified: identical to exp01 (2285/127 train, 414/23 val, 23 rows per class)")

all_df = pd.concat([train_df.assign(split="train"), val_df.assign(split="val")], ignore_index=True)

## Load the base model exactly as exp01 did, and capture the step-0 `score` head

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig, set_seed

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
print("padding_side:", tokenizer.padding_side, "| pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

set_seed(SEED)  # exp01 called set_seed(SEED) immediately before from_pretrained
base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=18,
    id2label=id2label,
    label2id=label2id,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    local_files_only=True,
)
base.config.pad_token_id = tokenizer.pad_token_id
base.eval()

W0 = base.score.weight.detach().float().cpu().clone()
print("score head:", tuple(W0.shape), "bias:", base.score.bias is not None)
print(f"step-0 score weight: mean={W0.mean():.5f} std={W0.std():.5f} "
      f"frob={W0.norm():.3f} initializer_range={base.config.initializer_range}")

In [ ]:
def pooled_forward(model, score_module, texts):
    # Returns the pooled final hidden state that feeds `score`, and the model's pooled logits.
    captured = {}
    handle = score_module.register_forward_hook(
        lambda mod, inp, out: captured.__setitem__("h", inp[0].detach()))
    hidden = np.zeros((len(texts), score_module_in_features), dtype=np.float32)
    logits = np.zeros((len(texts), 18), dtype=np.float32)
    try:
        with torch.no_grad():
            for start in range(0, len(texts), BATCH_SIZE):
                batch = texts[start:start + BATCH_SIZE]
                enc = tokenizer(batch, truncation=True, max_length=MAX_LENGTH,
                                padding=True, return_tensors="pt").to("cuda:0")
                with torch.autocast("cuda", dtype=torch.float16):
                    out = model(**enc).logits
                ids = enc["input_ids"]
                # Same pooling rule as transformers' *ForSequenceClassification: last non-pad token.
                non_pad = (ids != tokenizer.pad_token_id).int()
                tok_idx = torch.arange(ids.shape[-1], device=ids.device)
                last = (tok_idx * non_pad).argmax(-1)
                h = captured["h"][torch.arange(ids.shape[0], device=ids.device), last]
                hidden[start:start + len(batch)] = h.float().cpu().numpy()
                logits[start:start + len(batch)] = out.float().cpu().numpy()
    finally:
        handle.remove()
    return hidden, logits


def ce(logits, labels):
    return F.cross_entropy(torch.from_numpy(logits).double(),
                           torch.from_numpy(labels).long(), reduction="none").numpy()


score_module_in_features = W0.shape[1]
labels = all_df["labels"].to_numpy()
is_val = (all_df["split"] == "val").to_numpy()


def run_both_splits(model, score_module):
    # Val rows are run on their own, in val_df order with batch size 8, so every padded batch
    # has the same composition as exp01's trainer.predict(val_tok) (fp16 4-bit logits shift
    # slightly with batch padding). Train rows follow in train_df order.
    h_tr, l_tr = pooled_forward(model, score_module, train_df["text"].tolist())
    h_va, l_va = pooled_forward(model, score_module, val_df["text"].tolist())
    return np.concatenate([h_tr, h_va]), np.concatenate([l_tr, l_va])

## Pass 1 — step-0 model (base + random `score` head, adapter not attached)

In [ ]:
t0 = time.time()
H0, logits0_model = run_both_splits(base, base.score)
step0_minutes = (time.time() - t0) / 60
logits0 = H0 @ W0.numpy().T  # fp32 head, as after prepare_model_for_kbit_training
print(f"pass 1: {step0_minutes:.1f} min")
print("max |fp16 model logits - fp32 H0@W0^T|:", float(np.abs(logits0 - logits0_model).max()))

loss0 = ce(logits0, labels)
h0_norm = np.linalg.norm(H0, axis=1)
print(f"pooled hidden L2 norm: median={np.median(h0_norm):.1f} p5={np.percentile(h0_norm, 5):.1f} "
      f"p95={np.percentile(h0_norm, 95):.1f}")
print(f"step-0 logit std (per row, median): {np.median(logits0.std(axis=1)):.2f}")
print(f"step-0 CE: train mean={loss0[~is_val].mean():.2f} median={np.median(loss0[~is_val]):.2f} | "
      f"val mean={loss0[is_val].mean():.2f} | uniform ln(18)={np.log(18):.3f}")
print("step-0 val balanced accuracy:", balanced_accuracy_score(labels[is_val], logits0[is_val].argmax(1)))

rng = np.random.default_rng(SEED)
sim = []
for k in range(N_SIM_HEADS):
    Wk = rng.normal(0.0, float(W0.std()), size=W0.shape)
    lk = ce(H0 @ Wk.T, labels)
    sim.append({"draw": k, "train_mean_ce": float(lk[~is_val].mean()), "val_mean_ce": float(lk[is_val].mean())})
sim_df = pd.DataFrame(sim)
print(sim_df.describe().loc[["mean", "min", "max"]])

## Pass 2 — exp01 trained adapter + `score` head (eval mode, inference only)

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
model.eval()
print("trainable params in eval model:", sum(p.numel() for p in model.parameters() if p.requires_grad))
score_wrapper = model.base_model.model.score
W1 = score_wrapper.modules_to_save["default"].weight.detach().float().cpu().clone()
print("original (step-0) head preserved under adapter:",
      torch.equal(score_wrapper.original_module.weight.detach().float().cpu(), W0))
print(f"trained score weight: std={W1.std():.5f} frob={W1.norm():.3f} "
      f"(step-0 frob={W0.norm():.3f}, ratio={W1.norm() / W0.norm():.4f})")
cos = F.cosine_similarity(W0.flatten(), W1.flatten(), dim=0).item()
print(f"cosine(W0, W1) = {cos:.4f}")

t0 = time.time()
H1, logits1 = run_both_splits(model, score_wrapper)
trained_minutes = (time.time() - t0) / 60
print(f"pass 2: {trained_minutes:.1f} min")
print("max |model logits - H1@W1^T|:", float(np.abs(logits1 - H1 @ W1.numpy().T).max()))

loss1 = ce(logits1, labels)
pred1 = logits1.argmax(1)
val_bal_acc = balanced_accuracy_score(labels[is_val], pred1[is_val])
print(f"exp01 val balanced accuracy: {val_bal_acc:.4f} | val CE: {loss1[is_val].mean():.4f}")
print(f"exp01 train-split balanced accuracy: {balanced_accuracy_score(labels[~is_val], pred1[~is_val]):.4f} "
      f"| train CE: {loss1[~is_val].mean():.4f}")
print(f"trained logit std (per row, median): {np.median(logits1.std(axis=1)):.2f}")

## Save outputs

In [ ]:
logit_cols = [f"logit_{n}" for n in model_names]
meta = all_df[["id", "prompt_id", "model", "labels", "split"]].reset_index(drop=True)
pd.concat([meta, pd.DataFrame(logits1, columns=logit_cols)], axis=1).to_parquet(
    f"{OUTPUT_DIR}/exp01_logits.parquet", index=False)
pd.concat([meta, pd.DataFrame(logits0, columns=logit_cols)], axis=1).to_parquet(
    f"{OUTPUT_DIR}/step0_logits.parquet", index=False)
np.save(f"{OUTPUT_DIR}/pooled_hidden_step0.npy", H0.astype(np.float16))
np.save(f"{OUTPUT_DIR}/pooled_hidden_exp01.npy", H1.astype(np.float16))

diag = {
    "inference_only": True,
    "versions": versions,
    "adapter_dir": ADAPTER_DIR,
    "seed": SEED, "val_size": VAL_SIZE,
    "split": {"train_rows": len(train_df), "train_prompts": int(train_df["prompt_id"].nunique()),
              "val_rows": len(val_df), "val_prompts": int(val_df["prompt_id"].nunique())},
    "padding_side": tokenizer.padding_side,
    "initializer_range": base.config.initializer_range,
    "W0": {"std": float(W0.std()), "frob": float(W0.norm())},
    "W1": {"std": float(W1.std()), "frob": float(W1.norm()), "cosine_with_W0": cos},
    "pooled_hidden_norm": {"median": float(np.median(h0_norm)), "p5": float(np.percentile(h0_norm, 5)),
                           "p95": float(np.percentile(h0_norm, 95))},
    "pooled_hidden_norm_exp01_median": float(np.median(np.linalg.norm(H1, axis=1))),
    "step0": {"train_mean_ce": float(loss0[~is_val].mean()), "train_median_ce": float(np.median(loss0[~is_val])),
              "val_mean_ce": float(loss0[is_val].mean()), "val_median_ce": float(np.median(loss0[is_val])),
              "val_balanced_accuracy": float(balanced_accuracy_score(labels[is_val], logits0[is_val].argmax(1))),
              "median_row_logit_std": float(np.median(logits0.std(axis=1))),
              "sim_heads": sim},
    "exp01": {"val_balanced_accuracy": float(val_bal_acc), "val_mean_ce": float(loss1[is_val].mean()),
              "train_balanced_accuracy": float(balanced_accuracy_score(labels[~is_val], pred1[~is_val])),
              "train_mean_ce": float(loss1[~is_val].mean()),
              "median_row_logit_std": float(np.median(logits1.std(axis=1)))},
    "uniform_ce": float(np.log(18)),
    "runtime_min": {"step0_pass": step0_minutes, "exp01_pass": trained_minutes,
                    "total": (time.time() - RUN_START) / 60},
}
with open(f"{OUTPUT_DIR}/diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)
print(json.dumps(diag, indent=2))
print(sorted(os.listdir(OUTPUT_DIR)))

In [ ]:
# Confirms the session really had no internet access.
import urllib.request

try:
    urllib.request.urlopen("https://pypi.org", timeout=5)
    print("Internet: REACHABLE")
except Exception as e:
    print("Internet: unreachable ->", type(e).__name__)